In [ ]:
# ===============================
# STEP 1: Install & Import Libraries
# ===============================

!pip install geemap

import ee
import geemap

# Authenticate (ครั้งแรกเท่านั้น)
ee.Authenticate()

# Initialize GEE
ee.Initialize(project='ee-napaporn3401')

# Create map object
Map = geemap.Map()

print("Libraries loaded and GEE initialized successfully")

In [ ]:
# ===============================
# STEP 2: Define Area of Interest (AOI)
# ===============================

tak = ee.FeatureCollection("FAO/GAUL/2015/level1") \
        .filter(ee.Filter.eq('ADM1_NAME', 'Tak'))

# Center map
Map.centerObject(tak, 8)

# Display boundary
Map.addLayer(tak, {}, "Tak Province")

In [ ]:
# ===============================
# STEP 3: Topographic Data
# ===============================

# Digital Elevation Model
dem = ee.Image("USGS/SRTMGL1_003").clip(tak)

# Slope derived from DEM
slope = ee.Terrain.slope(dem)

# Visualization
Map.addLayer(dem, {'min': 0, 'max': 1000}, "Elevation")
Map.addLayer(slope, {'min': 0, 'max': 60}, "Slope")

In [ ]:
# ===============================
# STEP 4: Rainfall Data (ERA5)
# ===============================

rain = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterDate('2020-01-01', '2020-12-31') \
        .select("total_precipitation_sum") \
        .mean() \
        .clip(tak)

Map.addLayer(rain, {'min': 0, 'max': 0.01}, "Rainfall")

In [ ]:
# ===============================
# STEP 5: Distance to River
# ===============================

river = ee.Image("WWF/HydroSHEDS/15ACC").gt(100)

# Distance transform
dist = river.fastDistanceTransform().sqrt()

# Normalize (inverse: farther = safer)
dist_n_raw = ee.Image.constant(1).subtract(dist)

Map.addLayer(dist_n_raw, {'min': 0, 'max': 1}, "Distance to River (raw)")

In [ ]:
# ===============================
# STEP 6: Land Cover Data
# ===============================

landcover = ee.Image("MODIS/006/MCD12Q1/2020_01_01") \
              .select("LC_Type1") \
              .clip(tak)

Map.addLayer(landcover, {}, "Land Cover")

In [ ]:
# ===============================
# STEP 7: Normalization Function
# ===============================

def normalize(img):
    stats = img.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=tak,
        scale=5000,
        maxPixels=1e13
    )

    min_val = ee.Number(stats.values().get(0))
    max_val = ee.Number(stats.values().get(1))

    return img.subtract(min_val).divide(max_val.subtract(min_val))

In [ ]:
# ===============================
# STEP 8: Normalize Inputs
# ===============================

rain_n = normalize(rain)
slope_n = normalize(slope)
elev_n = normalize(dem)

# จาก STEP river
dist_n = normalize(dist_n_raw)

# landcover
lc_n = normalize(landcover)

In [ ]:
print(rain_n)
print(slope_n)
print(elev_n)
print(dist_n)
print(lc_n)

In [ ]:
# ===============================
# STEP 9: Flood Risk Model
# ===============================

flood_risk = (
    rain_n.multiply(0.30)
    .add(slope_n.multiply(0.25))
    .add(elev_n.multiply(0.20))
    .add(dist_n.multiply(0.15))
    .add(lc_n.multiply(0.10))
)

Map.addLayer(
    flood_risk,
    {'min': 0, 'max': 1, 'palette': ['green','yellow','red']},
    "Flood Risk (Continuous)"
)

In [ ]:
# ===============================
# STEP 10: Risk Classification
# ===============================

risk_class = flood_risk.expression(
    "(b(0) < 0.25) ? 1" +
    ": (b(0) < 0.45) ? 2" +
    ": 3"
)

Map.addLayer(
    risk_class,
    {'min':1, 'max':3, 'palette':['green','yellow','red']},
    "Risk Class"
)

In [ ]:
dem = dem.clip(tak)
slope = slope.clip(tak)
rain = rain.clip(tak)
dist_n_raw = dist_n_raw.clip(tak)
landcover = landcover.clip(tak)

In [ ]:
# ===============================
# STEP 11: Layer Control
# ===============================

Map.addLayerControl()

Map

In [ ]:
import matplotlib.pyplot as plt

weights = [0.24, 0.30, 0.36]
risk_mean = [0.42, 0.47, 0.52]  # ตัวอย่าง (คำนวณจริงควรใช้ reduceRegion)

plt.plot(weights, risk_mean)
plt.xlabel("Rainfall Weight")
plt.ylabel("Average Flood Risk")
plt.title("Sensitivity Analysis")
plt.show()